In [1]:
import sys
print(sys.executable)

c:\Users\EL077\Desktop\git-projects\Azure-OpenAI-kma-Weather-bot\.venv\Scripts\python.exe


# 4 get_korea_weather 관련 함수 

In [ ]:
# =============================
# 기본 세팅: 상수처럼 파일 상단에 둔다. 
# =============================
# 1. 라이브러리 설치
import os
import math
import requests
from datetime import datetime, timedelta


In [ ]:
#--------------
# 1. 강수 형태 매핑을 위한 딕셔너리
# --------------
PTY_CODE = {
    "0": "강수 없음",
    "1": "비",
    "2": "비/눈",
    "3": "눈",
    "5": "빗방울",
    "6": "빗방울/눈날림",
    "7": "눈날림"
}

# 지역별 위경도 매핑 > 추후 Geocoding
LOCATION_COORDS = {
    "서울": (37.5665, 126.9780),
    "부산": (35.1796, 129.0756),
    "대구": (35.8714, 128.6014),
    "인천": (37.4563, 126.7052),
    "광주": (35.1595, 126.8526),
    "대전": (36.3504, 127.3845),
    "울산": (35.5384, 129.3114),
    "세종": (36.4800, 127.2890),
    "제주": (33.4996, 126.5312),
}

In [ ]:
# [1] 기상청 초단기 실황 api에 넣을 기준 날짜와 기준 시간을 자동으로 만들어주는 함수 
# ===============================
def get_base_time_for_ncst():
    now = datetime.now()
    # --------------------------------
    # 초단기실황은 정시 기준 자료라, 안정적인 데이터 로드를 위해 너무 이른 시각이면 이전 시간 사용
    # ---------------------------------
    if now.minute < 40:
        now = now - timedelta(hours=1)

    return now.strftime("%Y%m%d"), now.strftime("%H00")


In [ ]:

# [2] 위도, 경도를 기상청 전용 격자 좌표로 바꿔주는 함수 convert_lat_lon_to_grid(lat, lon):
# ===============================
def convert_lat_lon_to_grid(lat, lon):
    # 지구 반지름
    RE = 6371.00877

    # 격자 간격 5km
    GRID = 5.0

    # 투영 계산에 쓰는 기준 위도
    SLAT1 = 30.0
    SLAT2 = 60.0

    # 기준 경도
    OLON = 126.0

    # 기준 위도
    OLAT = 38.0

    # 기준점의 격자 좌표 보정값
    XO = 43
    YO = 136

    # 도 단위를 라디언으로 바꿈
    DEGRAD = math.pi / 180.0

    # 지구 반지름을 격자 단위로 환산 -> 지구 반지름 km /  격자 간격 5km
    re = RE / GRID

    # 투영 계산용 보정값 (지구는 둥그니까 지도로 펼칠 때 생기는 왜곡 보정)
    slat1 = SLAT1 * DEGRAD
    slat2 = SLAT2 * DEGRAD
    olon = OLON * DEGRAD
    olat = OLAT * DEGRAD

    sn = math.tan(math.pi * 0.25 + slat2 * 0.5) / math.tan(math.pi * 0.25 + slat1 * 0.5)
    sn = math.log(math.cos(slat1) / math.cos(slat2)) / math.log(sn)

    sf = math.tan(math.pi * 0.25 + slat1 * 0.5)
    sf = (sf ** sn) * math.cos(slat1) / sn

    ro = math.tan(math.pi * 0.25 + olat * 0.5)
    ro = re * sf / (ro ** sn)

    # 입력된 위도/경도를 위치로 변환
    ra = math.tan(math.pi * 0.25 + lat * DEGRAD * 0.5)
    ra = re * sf / (ra ** sn)

    # 경도 차이 계산
    theta = lon * DEGRAD - olon

    if theta > math.pi:
        theta -= 2.0 * math.pi
    if theta < -math.pi:
        theta += 2.0 * math.pi

    theta *= sn

    x = int(ra * math.sin(theta) + XO + 0.5)
    y = int(ro - ra * math.cos(theta) + YO + 0.5)

    return x, y



In [ ]:
#  [3] 날씨 요청 함수 get_korea_weather(location=None, latitude=None, longitude=None)
# ===============================
def get_korea_weather(location=None, latitude=None, longitude=None):

    # 엔드포인트, url, 기상청 격자좌표, 시각 불러오기
    weather_api_key = os.getenv("WEATHER_API_KEY")
    url = os.getenv("GET_KOREA_WEATHER")
    nx, ny = convert_lat_lon_to_grid(latitude, longitude)
    base_date, base_time = get_base_time_for_ncst()
    
    # request 형식
    params = {
        "serviceKey": weather_api_key,
        "pageNo": 1,
        "numOfRows": 100,
        "dataType": "JSON",
        "base_date": base_date,
        "base_time": base_time,
        "nx": nx,
        "ny": ny
    }

    #------------
    # ?????????????요청하기
    #------------  
    response = requests.get(url, params=params)
    print("status_code:", response.status_code)
    # print("response preview:", response.text[:300]) # response의 모든 정보 

    data = response.json()

    # 헤더 정보 및 기상청 api 오류 발생 확인 코드
    header = data["response"]["header"]
    if header["resultCode"] != "00":
        return f"기상청 API 오류: {header['resultCode']} / {header['resultMsg']}"

    # response 중 기온, 습도, 강수형태, 1시간 강수량, 풍속 정보만 꺼내기
    items = data["response"]["body"]["items"]["item"]

    weather_info = {}
    for item in items:
        weather_info[item["category"]] = item["obsrValue"]

    # 기온
    temp = weather_info.get("T1H")  

    # 습도
    humidity = weather_info.get("REH")  

    # 강수형태
    rain_type = PTY_CODE.get(weather_info.get("PTY"), weather_info.get("PTY"))  

    # 1시간 강수량
    rain_1h = weather_info.get("RN1")  

    # 풍속
    wind_speed = weather_info.get("WSD")  

    return (
        f"{location}의 기상청 초단기실황입니다. "
        f"기온은 {temp}℃, 습도는 {humidity}%, 강수형태는 {rain_type}, "
        f"1시간 강수량은 {rain_1h}mm, 풍속은 {wind_speed}m/s입니다. "
        f"조회 기준시각은 {base_date} {base_time}, 격자좌표는 nx={nx}, ny={ny}입니다."
    )



In [ ]:
# demo
#=======================
demo = get_korea_weather(
    location="부산",
    latitude=35.1796,
    longitude=129.0756)
print(demo)

# --------------
# 출력결과
# --------------
# status_code: 200
# response preview: {"response":{
#     "header":{"resultCode":"00","resultMsg":"NORMAL_SERVICE"},
#     "body":{
#         "dataType":"JSON",
#         "items":{
#             "item":[
#                 {"baseDate":"20260606","baseTime":"0000","category":"PTY","nx":98,"ny":76,"obsrValue":"0"}, # 강수형태
#                 {"baseDate":"20260606","baseTime":"0000","category":"REH","nx":98,"ny":76,"obsrValue":"41"}, # 습도
#                 {"baseDate":"20260606","baseTime":"0000","category":"RN1","nx":98,"ny":76,"obsrValue":"0"}, # 1시간 강수량
#                 {"baseDate":"20260606","baseTime":"0000","category":"T1H","nx":98,"ny":76,"obsrValue":"18.6"}, # 기온
#                 {"baseDate":"20260606","baseTime":"0000","category":"UUU","nx":98,"ny":76,"obsrValue":"0"},
#                 {"baseDate":"20260606","baseTime":"0000","category":"VEC","nx":98,"ny":76,"obsrValue":"7"},
#                 {"baseDate":"20260606","baseTime":"0000","category":"VVV","nx":98,"ny":76,"obsrValue":"-0.7"},
#                 {"baseDate":"20260606","baseTime":"0000","category":"WSD","nx":98,"ny":76,"obsrValue":"0.8"}]}, # 풍속
#     "pageNo":1,
#     "numOfRows":100,
#     "totalCount":8}}}
# 부산의 기상청 초단기실황입니다. 기온은 18.6℃, 습도는 41%, 강수형태는 강수 없음, 1시간 강수량은 0mm, 풍속은 0.8m/s입니다. 조회 기준시각은 20260606 0000, 격자좌표는 nx=98, ny=76입니다.

status_code: 200
response preview: {"response":{"header":{"resultCode":"00","resultMsg":"NORMAL_SERVICE"},"body":{"dataType":"JSON","items":{"item":[{"baseDate":"20260606","baseTime":"0000","category":"PTY","nx":98,"ny":76,"obsrValue":"0"},{"baseDate":"20260606","baseTime":"0000","category":"REH","nx":98,"ny":76,"obsrValue":"41"},{"baseDate":"20260606","baseTime":"0000","category":"RN1","nx":98,"ny":76,"obsrValue":"0"},{"baseDate":"20260606","baseTime":"0000","category":"T1H","nx":98,"ny":76,"obsrValue":"18.6"},{"baseDate":"20260606","baseTime":"0000","category":"UUU","nx":98,"ny":76,"obsrValue":"0"},{"baseDate":"20260606","baseTime":"0000","category":"VEC","nx":98,"ny":76,"obsrValue":"7"},{"baseDate":"20260606","baseTime":"0000","category":"VVV","nx":98,"ny":76,"obsrValue":"-0.7"},{"baseDate":"20260606","baseTime":"0000","category":"WSD","nx":98,"ny":76,"obsrValue":"0.8"}]},"pageNo":1,"numOfRows":100,"totalCount":8}}}
부산의 기상청 초단기실황입니다. 기온은 18.6℃, 습도는 41%, 강수형태는 강수 없음, 1시간 강수량은 0mm, 풍속